In [1]:
import os
import pathlib
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
from dotenv import load_dotenv

# Locate project root and load env variables
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")

RAW = pathlib.Path(ROOT / os.getenv('DATA_DIR_RAW', 'data/raw'))
PROC = pathlib.Path(ROOT / os.getenv('DATA_DIR_PROCESSED', 'data/processed'))

RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)

print('RAW Directory ->', RAW.resolve())
print('PROC Directory ->', PROC.resolve())

RAW Directory -> /home/emb9640/Bootcamp_Eric_Bruckenstein/homework/homework05/data/raw
PROC Directory -> /home/emb9640/Bootcamp_Eric_Bruckenstein/homework/homework05/data/processed


In [2]:
# 1. Create a Synthetic Sample DataFrame (PLTR)
dates = pd.date_range('2025-01-01', periods=30, freq='D')
df = pd.DataFrame({
    'date': dates, 
    'ticker': ['PLTR'] * 30, 
    'price': 40 + np.random.randn(30).cumsum(),
    'volume': np.random.randint(10000, 50000, size=30)
})

print("Sample Data Created:")
display(df.head())

Sample Data Created:


,date,ticker,price,volume
0,2025-01-01,PLTR,40.640166,38257
1,2025-01-02,PLTR,40.599069,32530
2,2025-01-03,PLTR,40.687279,21041
3,2025-01-04,PLTR,40.275371,25998
4,2025-01-05,PLTR,39.838856,46380


In [3]:
# 2. Save CSV to data/raw/ and Parquet to data/processed/
timestamp = dt.datetime.now().strftime('%Y%m%d-%H%M')

# Save CSV (Raw)
csv_path = RAW / f"pltr_sample_{timestamp}.csv"
df.to_csv(csv_path, index=False)
print(f"Saved CSV to: {csv_path}")

# Save Parquet (Processed)
pq_path = PROC / f"pltr_sample_{timestamp}.parquet"
try:
    df.to_parquet(pq_path)
    print(f"Saved Parquet to: {pq_path}")
except Exception as e:
    print('Parquet engine not available. Install pyarrow to complete this step.')
    pq_path = None

Saved CSV to: /home/emb9640/Bootcamp_Eric_Bruckenstein/homework/homework05/data/raw/pltr_sample_20260818-0007.csv
Saved Parquet to: /home/emb9640/Bootcamp_Eric_Bruckenstein/homework/homework05/data/processed/pltr_sample_20260818-0007.parquet


In [4]:
# 3. Reload and Validate Data integrity
def validate_loaded(original: pd.DataFrame, reloaded: pd.DataFrame, source: str):
    """Checks shape and required dtypes of loaded data."""
    checks = {
        'shape_equal': original.shape == reloaded.shape,
        'date_is_datetime': pd.api.types.is_datetime64_any_dtype(reloaded['date']) if 'date' in reloaded.columns else False,
        'price_is_numeric': pd.api.types.is_numeric_dtype(reloaded['price']) if 'price' in reloaded.columns else False,
    }
    
    print(f"\n--- Validation Report for {source} ---")
    for check, passed in checks.items():
        print(f"[{'Pass' if passed else 'Fail'}] {check}")
    return all(checks.values())

# Validate CSV
df_csv = pd.read_csv(csv_path, parse_dates=['date'])
validate_loaded(df, df_csv, "CSV")

# Validate Parquet
if pq_path:
    try:
        df_pq = pd.read_parquet(pq_path)
        validate_loaded(df, df_pq, "Parquet")
    except Exception as e:
        print('Parquet read failed:', e)


--- Validation Report for CSV ---
[Pass] shape_equal
[Pass] date_is_datetime
[Pass] price_is_numeric

--- Validation Report for Parquet ---
[Pass] shape_equal
[Pass] date_is_datetime
[Pass] price_is_numeric


In [5]:
# 4. Reusable Storage Utilities
import typing as t

def detect_format(path: t.Union[str, pathlib.Path]) -> str:
    s = str(path).lower()
    if s.endswith('.csv'): return 'csv'
    if s.endswith('.parquet') or s.endswith('.pq') or s.endswith('.parq'): return 'parquet'
    raise ValueError('Unsupported format: ' + s)

def write_df(df: pd.DataFrame, path: t.Union[str, pathlib.Path]) -> pathlib.Path:
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    fmt = detect_format(p)
    
    if fmt == 'csv':
        df.to_csv(p, index=False)
    else:
        try:
            df.to_parquet(p)
        except Exception as e:
            raise RuntimeError('Parquet engine missing. Run: pip install pyarrow') from e
    return p

def read_df(path: t.Union[str, pathlib.Path], date_cols: t.List[str] = None) -> pd.DataFrame:
    p = pathlib.Path(path)
    if not p.exists():
        raise FileNotFoundError(f"File not found: {p}")
        
    fmt = detect_format(p)
    if fmt == 'csv':
        return pd.read_csv(p, parse_dates=date_cols if date_cols else False)
    else:
        try:
            return pd.read_parquet(p)
        except Exception as e:
            raise RuntimeError('Parquet engine missing. Run: pip install pyarrow') from e

print("\n--- Testing write_df / read_df ---")
util_csv = RAW / f"util_demo_{timestamp}.csv"
util_pq = PROC / f"util_demo_{timestamp}.parquet"

write_df(df, util_csv)
print(f"write_df (CSV): Success -> {util_csv}")
df_reloaded_csv = read_df(util_csv, date_cols=['date'])
print("read_df (CSV): Data validated:", df_reloaded_csv.shape == df.shape)

try:
    write_df(df, util_pq)
    print(f"write_df (Parquet): Success -> {util_pq}")
    df_reloaded_pq = read_df(util_pq)
    print("read_df (Parquet): Data validated:", df_reloaded_pq.shape == df.shape)
except RuntimeError as e:
    print('Skipping Parquet util demo:', e)


--- Testing write_df / read_df ---
write_df (CSV): Success -> /home/emb9640/Bootcamp_Eric_Bruckenstein/homework/homework05/data/raw/util_demo_20260818-0007.csv
read_df (CSV): Data validated: True
write_df (Parquet): Success -> /home/emb9640/Bootcamp_Eric_Bruckenstein/homework/homework05/data/processed/util_demo_20260818-0007.parquet
read_df (Parquet): Data validated: True
